In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone 1
This milestone has been created to familiarize you with the process of Exploratory Data     Analysis (EDA), text processing, and establishing baseline similarity metrics required  specifically for Natural Language Processing (NLP) pipelines.  

In [2]:
import pandas as pd
import numpy as np
import string

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)

from sklearn.metrics.pairwise import cosine_similarity

In [3]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print(train.shape)
train.head()

(2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
display(train.info())

display(train.isnull().sum())

display(train.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


None

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


**Q1) Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?**

In [5]:
answer_counts = train["answer"].value_counts()

print(answer_counts)

most_freq = answer_counts.max()
least_freq = answer_counts.min()

q1_answer = most_freq + least_freq

print("Answer:", q1_answer)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Answer: 814


**Q2) After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?**

In [6]:
translator = str.maketrans('', '', string.punctuation)

all_words = []

for text in train["prompt"]:
    
    cleaned = text.lower().translate(translator)
    
    words = cleaned.split()
    
    all_words.extend(words)

vocab = set(all_words)

q2_answer = len(vocab)

print( q2_answer)

859


**Q3) Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?**

In [7]:
row1 = train.loc[train["id"] == 1, "prompt"].values[0]

cleaned = row1.lower().translate(translator)

tokens = cleaned.split()

filtered = [
    w
    for w in tokens
    if w not in ENGLISH_STOP_WORDS
]

print(filtered)

q3_answer = len(filtered)

print( q3_answer)

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
13


**Q4) Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?**

In [8]:
combined_text = (
    train["prompt"].astype(str)
    + " "
    + train["A"].astype(str)
    + " "
    + train["B"].astype(str)
    + " "
    + train["C"].astype(str)
    + " "
    + train["D"].astype(str)
    + " "
    + train["E"].astype(str)
)

vectorizer = TfidfVectorizer(
    stop_words="english"
)

vectorizer.fit(combined_text)

q4_answer = len(vectorizer.get_feature_names_out())

print("TF-IDF Vocabulary Size:", q4_answer)

TF-IDF Vocabulary Size: 2762


**Q5) Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).**

In [9]:
row1 = train.loc[train["id"] == 1].iloc[0]

prompt_vec = vectorizer.transform([row1["prompt"]])

optionA_vec = vectorizer.transform([row1["A"]])

similarity = cosine_similarity(
    prompt_vec,
    optionA_vec
)[0][0]

print(round(similarity, 4))

0.272


**Q6) Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.**

In [10]:
labels = ["A","B","C","D","E"]

correct = 0

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    scores = {}

    for label in labels:

        option_vec = vectorizer.transform([row[label]])

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        scores[label] = sim

    pred = max(scores, key=scores.get)

    if pred == row["answer"]:
        correct += 1

accuracy = 100 * correct / len(train)

print(round(accuracy,1))

13.6


**Q7) If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?**

In [11]:
truth = "C"
preds = ["C","A","B"]

if truth in preds:
    score = 1/(preds.index(truth)+1)
else:
    score = 0

print(score)

1.0


**Q8) If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?**

In [12]:
truth = "B"
preds = ["D","B","E"]

if truth in preds:
    score = 1/(preds.index(truth)+1)
else:
    score = 0

print(score)

0.5


**Q9) The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?**

In [13]:
def map3_score(actual, preds):

    if actual in preds:
        return 1 / (preds.index(actual) + 1)

    return 0

In [14]:
answer_counts = train["answer"].value_counts()

top3 = answer_counts.index[:3].tolist()

print(top3)

['B', 'C', 'A']


In [15]:
scores = []

for actual in train["answer"]:

    scores.append(
        map3_score(
            actual,
            top3
        )
    )

q9_answer = np.mean(scores)

print(round(q9_answer,4))

0.4212


**Q10) The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?**

In [16]:
scores = []

labels = ["A","B","C","D","E"]

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    sims = {}

    for label in labels:

        option_vec = vectorizer.transform([row[label]])

        sims[label] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

    ranked = sorted(
        sims,
        key=sims.get,
        reverse=True
    )

    top3 = ranked[:3]

    scores.append(
        map3_score(
            row["answer"],
            top3
        )
    )

final_map3 = np.mean(scores)

print(round(final_map3,4))

0.2962
